# 🗺️ Analyse Géomatique Complète

Ce notebook effectue une **analyse géomatique complète** pour un point géographique donné :

- 📍 Affichage des coordonnées sur carte interactive
- 🌧️ Taux de précipitation
- 💨 Rose des vents (direction et vitesse)
- 🌡️ Température
- 🌊 Risque d'inondation
- 🏔️ Risque de glissement de terrain
- 🌍 Risque sismique

**Sources de données :**
- [Open-Meteo](https://open-meteo.com/) – météo historique et prévisions (gratuit, sans clé API)
- [Open-Elevation](https://open-elevation.com/) – altitude du terrain (gratuit)
- [USGS Earthquake API](https://earthquake.usgs.gov/) – données sismiques (gratuit)
- [OpenTopoData](https://www.opentopodata.org/) – données topographiques (gratuit)


## 1. Installation des bibliothèques

In [ ]:
!pip install folium requests pandas numpy matplotlib plotly scipy -q

## 2. Importation des bibliothèques

In [ ]:
import requests
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import display, HTML
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliothèques importées avec succès')

## 3. 📍 Définition des coordonnées du site

Modifiez `LATITUDE` et `LONGITUDE` selon le site à étudier.
Vous pouvez aussi indiquer un nom de site pour les rapports.

In [ ]:
# ============================================================
# PARAMÈTRES À MODIFIER
# ============================================================
LATITUDE  = 33.5731  # Latitude du site (ex: Casablanca, Maroc)
LONGITUDE = -7.5898  # Longitude du site
NOM_SITE  = 'Casablanca, Maroc'

# Période d'analyse météo (format AAAA-MM-JJ)
DATE_DEBUT = '2024-01-01'
DATE_FIN   = '2024-12-31'
# ============================================================

print(f'📍 Site d\'étude : {NOM_SITE}')
print(f'   Latitude  : {LATITUDE}°')
print(f'   Longitude : {LONGITUDE}°')
print(f'   Période   : {DATE_DEBUT} → {DATE_FIN}')

## 4. 🗺️ Affichage des coordonnées sur carte interactive

In [ ]:
def creer_carte(lat, lon, nom_site):
    """Crée une carte interactive Folium avec plusieurs fonds de carte."""
    carte = folium.Map(
        location=[lat, lon],
        zoom_start=12,
        tiles='OpenStreetMap'
    )

    # Ajout de couches de fond supplémentaires
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite Esri', overlay=False, control=True
    ).add_to(carte)

    folium.TileLayer(
        'https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='OpenTopoMap', name='Topographique', overlay=False, control=True
    ).add_to(carte)

    # Marqueur principal
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(
            f'<b>{nom_site}</b><br>Lat: {lat:.4f}°<br>Lon: {lon:.4f}°',
            max_width=200
        ),
        tooltip=nom_site,
        icon=folium.Icon(color='red', icon='home', prefix='fa')
    ).add_to(carte)

    # Cercle de rayon de 1 km autour du site
    folium.Circle(
        location=[lat, lon],
        radius=1000,
        color='blue',
        fill=True,
        fill_opacity=0.1,
        popup='Zone d\'étude (1 km)'
    ).add_to(carte)

    # Contrôle des couches
    folium.LayerControl().add_to(carte)

    return carte


carte_principale = creer_carte(LATITUDE, LONGITUDE, NOM_SITE)
carte_principale.save('/tmp/carte_site.html')
display(carte_principale)
print('✅ Carte sauvegardée dans /tmp/carte_site.html')

## 5. 🌦️ Récupération des données météorologiques (Open-Meteo)

In [ ]:
def recuperer_donnees_meteo(lat, lon, date_debut, date_fin):
    """
    Récupère les données météo historiques via l'API Open-Meteo (gratuite).
    Variables : précipitations, température, vent.
    """
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_debut,
        'end_date': date_fin,
        'daily': [
            'temperature_2m_max',
            'temperature_2m_min',
            'temperature_2m_mean',
            'precipitation_sum',
            'wind_speed_10m_max',
            'wind_direction_10m_dominant',
            'et0_fao_evapotranspiration',
        ],
        'timezone': 'auto'
    }

    print('⏳ Téléchargement des données météo en cours...')
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data['daily'])
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df.columns = [
        'temp_max', 'temp_min', 'temp_mean',
        'precipitation', 'vent_vitesse', 'vent_direction',
        'evapotranspiration'
    ]
    print(f'✅ {len(df)} jours de données récupérés ({date_debut} → {date_fin})')
    return df


df_meteo = recuperer_donnees_meteo(LATITUDE, LONGITUDE, DATE_DEBUT, DATE_FIN)
print('\nAperçu des données :')
df_meteo.head()

## 6. 🌧️ Analyse des précipitations

In [ ]:
def analyser_precipitations(df):
    """Analyse et visualise les données de précipitation."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('🌧️ Analyse des Précipitations', fontsize=16, fontweight='bold')

    precip = df['precipitation'].fillna(0)

    # 1. Précipitations journalières
    axes[0, 0].bar(df.index, precip, color='steelblue', alpha=0.7, width=1)
    axes[0, 0].set_title('Précipitations journalières (mm)')
    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('Précipitation (mm)')
    axes[0, 0].tick_params(axis='x', rotation=45)

    # 2. Précipitations mensuelles cumulées
    precip_mensuel = precip.resample('ME').sum()
    mois = [d.strftime('%b %Y') for d in precip_mensuel.index]
    axes[0, 1].bar(range(len(mois)), precip_mensuel.values, color='royalblue', alpha=0.8)
    axes[0, 1].set_xticks(range(len(mois)))
    axes[0, 1].set_xticklabels(mois, rotation=45, ha='right')
    axes[0, 1].set_title('Précipitations mensuelles cumulées (mm)')
    axes[0, 1].set_ylabel('Précipitation (mm)')

    # 3. Distribution des précipitations
    jours_pluie = precip[precip > 0]
    if len(jours_pluie) > 0:
        axes[1, 0].hist(jours_pluie, bins=30, color='deepskyblue', edgecolor='black', alpha=0.7)
        axes[1, 0].set_title('Distribution des jours de pluie')
        axes[1, 0].set_xlabel('Précipitation (mm)')
        axes[1, 0].set_ylabel('Nombre de jours')
        axes[1, 0].axvline(jours_pluie.mean(), color='red', linestyle='--',
                           label=f'Moyenne: {jours_pluie.mean():.1f} mm')
        axes[1, 0].legend()

    # 4. Statistiques
    stats = {
        'Total annuel (mm)': precip.sum(),
        'Moyenne journalière (mm)': precip.mean(),
        'Maximum journalier (mm)': precip.max(),
        'Jours de pluie': (precip > 0).sum(),
        'Jours > 10 mm': (precip > 10).sum(),
        'Jours > 30 mm (extrêmes)': (precip > 30).sum(),
    }
    axes[1, 1].axis('off')
    y_pos = 0.9
    axes[1, 1].text(0.1, 1.0, '📊 Statistiques des précipitations',
                    fontsize=12, fontweight='bold', transform=axes[1, 1].transAxes)
    for label, valeur in stats.items():
        axes[1, 1].text(0.1, y_pos, f'{label}: {valeur:.1f}',
                        fontsize=11, transform=axes[1, 1].transAxes)
        y_pos -= 0.13

    plt.tight_layout()
    plt.savefig('/tmp/precipitations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Analyse des précipitations terminée')
    return stats


stats_precip = analyser_precipitations(df_meteo)

## 7. 💨 Rose des Vents (Direction et Vitesse)

In [ ]:
def tracer_rose_des_vents(df):
    """Trace la rose des vents avec direction et vitesse."""
    directions = df['vent_direction'].dropna().values
    vitesses = df['vent_vitesse'].dropna().values

    # Aligner les longueurs
    mask = ~(np.isnan(df['vent_direction']) | np.isnan(df['vent_vitesse']))
    directions = df.loc[mask, 'vent_direction'].values
    vitesses = df.loc[mask, 'vent_vitesse'].values

    # Catégories de vitesse (km/h)
    bins_vitesse = [0, 10, 20, 30, 40, 100]
    labels_vitesse = ['0–10', '10–20', '20–30', '30–40', '>40']
    couleurs = ['#91BFDB', '#4575B4', '#FEE090', '#F46D43', '#A50026']

    # Secteurs directionnels (16 points cardinaux)
    nb_secteurs = 16
    angles = np.linspace(0, 360, nb_secteurs, endpoint=False)
    largeur = 360 / nb_secteurs

    fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                              subplot_kw={'projection': None})

    # Rose des vents polaire
    ax_rose = plt.subplot(1, 2, 1, projection='polar')
    ax_rose.set_theta_zero_location('N')
    ax_rose.set_theta_direction(-1)

    # Calculer la fréquence par secteur et catégorie de vitesse
    freq_cumul = np.zeros(nb_secteurs)
    for i, (v_min, v_max, label, coul) in enumerate(
            zip(bins_vitesse[:-1], bins_vitesse[1:], labels_vitesse, couleurs)):
        freq = np.zeros(nb_secteurs)
        for j, angle in enumerate(angles):
            masque_dir = (directions >= angle - largeur / 2) & (directions < angle + largeur / 2)
            masque_vit = (vitesses >= v_min) & (vitesses < v_max)
            freq[j] = np.sum(masque_dir & masque_vit) / len(directions) * 100
        bars = ax_rose.bar(
            np.radians(angles), freq,
            width=np.radians(largeur),
            bottom=freq_cumul,
            color=coul, alpha=0.85, label=f'{label} km/h'
        )
        freq_cumul += freq

    ax_rose.set_title('Rose des Vents\n(% du temps par direction et vitesse)',
                       pad=20, fontsize=12, fontweight='bold')
    ax_rose.set_xticks(np.radians([0, 45, 90, 135, 180, 225, 270, 315]))
    ax_rose.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SO', 'O', 'NO'])
    ax_rose.legend(loc='lower left', bbox_to_anchor=(1.05, 0), title='Vitesse')

    # Histogramme des vitesses
    ax_hist = plt.subplot(1, 2, 2)
    ax_hist.hist(vitesses, bins=25, color='steelblue', edgecolor='black', alpha=0.75)
    ax_hist.axvline(vitesses.mean(), color='red', linestyle='--',
                    label=f'Moyenne: {vitesses.mean():.1f} km/h')
    ax_hist.axvline(np.percentile(vitesses, 90), color='orange', linestyle='--',
                    label=f'P90: {np.percentile(vitesses, 90):.1f} km/h')
    ax_hist.set_title('Distribution des Vitesses de Vent', fontsize=12, fontweight='bold')
    ax_hist.set_xlabel('Vitesse (km/h)')
    ax_hist.set_ylabel('Nombre de jours')
    ax_hist.legend()

    # Statistiques
    stats_vent = {
        'Vitesse moyenne': f'{vitesses.mean():.1f} km/h',
        'Vitesse max': f'{vitesses.max():.1f} km/h',
        'Vitesse P90': f'{np.percentile(vitesses, 90):.1f} km/h',
    }
    textstr = '\n'.join([f'{k}: {v}' for k, v in stats_vent.items()])
    ax_hist.text(0.98, 0.95, textstr, transform=ax_hist.transAxes,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.suptitle(f'💨 Analyse des Vents – {NOM_SITE}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/tmp/rose_des_vents.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Rose des vents générée')


tracer_rose_des_vents(df_meteo)

## 8. 🌡️ Analyse des Températures

In [ ]:
def analyser_temperatures(df):
    """Analyse et visualise les données de température."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('🌡️ Analyse des Températures', fontsize=16, fontweight='bold')

    # 1. Courbe de température (min/moy/max)
    ax = axes[0, 0]
    ax.fill_between(df.index, df['temp_min'], df['temp_max'],
                    alpha=0.2, color='orange', label='Plage min–max')
    ax.plot(df.index, df['temp_mean'], color='red', linewidth=1.5, label='Température moyenne')
    ax.plot(df.index, df['temp_max'], color='tomato', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.plot(df.index, df['temp_min'], color='steelblue', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_title('Températures journalières (°C)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Température (°C)')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()

    # 2. Températures mensuelles moyennes
    temp_mensuel = df[['temp_min', 'temp_mean', 'temp_max']].resample('ME').mean()
    mois = [d.strftime('%b') for d in temp_mensuel.index]
    x = np.arange(len(mois))
    ax2 = axes[0, 1]
    ax2.fill_between(x, temp_mensuel['temp_min'], temp_mensuel['temp_max'],
                     alpha=0.3, color='orange')
    ax2.plot(x, temp_mensuel['temp_mean'], 'r-o', linewidth=2, label='T moy')
    ax2.plot(x, temp_mensuel['temp_max'], 'r--', linewidth=1, alpha=0.6, label='T max')
    ax2.plot(x, temp_mensuel['temp_min'], 'b--', linewidth=1, alpha=0.6, label='T min')
    ax2.set_xticks(x)
    ax2.set_xticklabels(mois)
    ax2.set_title('Températures mensuelles moyennes (°C)')
    ax2.set_ylabel('Température (°C)')
    ax2.legend()

    # 3. Heatmap journalière (mois × jour)
    temp_pivot = df['temp_mean'].groupby([df.index.month, df.index.day]).mean().unstack()
    im = axes[1, 0].imshow(temp_pivot.values, aspect='auto', cmap='RdYlBu_r')
    axes[1, 0].set_title('Carte thermique (mois × jour)')
    axes[1, 0].set_xlabel('Jour du mois')
    axes[1, 0].set_ylabel('Mois')
    axes[1, 0].set_yticks(range(12))
    axes[1, 0].set_yticklabels(['Jan','Fév','Mar','Avr','Mai','Jun',
                                 'Jul','Aoû','Sep','Oct','Nov','Déc'])
    plt.colorbar(im, ax=axes[1, 0], label='°C')

    # 4. Statistiques thermiques
    axes[1, 1].axis('off')
    temp_mean = df['temp_mean'].dropna()
    stats_temp = [
        ('Température annuelle moyenne', f'{temp_mean.mean():.1f} °C'),
        ('Température maximale', f'{df["temp_max"].max():.1f} °C'),
        ('Température minimale', f'{df["temp_min"].min():.1f} °C'),
        ('Amplitude annuelle', f'{df["temp_max"].max() - df["temp_min"].min():.1f} °C'),
        ('Jours > 35°C', f'{(df["temp_max"] > 35).sum()}'),
        ('Jours < 0°C', f'{(df["temp_min"] < 0).sum()}'),
    ]
    axes[1, 1].text(0.1, 1.0, '📊 Statistiques thermiques',
                    fontsize=12, fontweight='bold', transform=axes[1, 1].transAxes)
    y_pos = 0.85
    for label, val in stats_temp:
        axes[1, 1].text(0.1, y_pos, f'{label}: {val}',
                        fontsize=11, transform=axes[1, 1].transAxes)
        y_pos -= 0.13

    plt.tight_layout()
    plt.savefig('/tmp/temperatures.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Analyse des températures terminée')


analyser_temperatures(df_meteo)

## 9. 🏔️ Récupération des données d'altitude (OpenTopoData)

In [ ]:
def recuperer_altitude(lat, lon, rayon_km=5, nb_points=25):
    """
    Récupère les altitudes d'une grille de points autour du site
    via l'API OpenTopoData (SRTM 90m, gratuit).
    """
    # Génération d'une grille de points
    delta_deg = rayon_km / 111.0  # ~111 km par degré
    lats = np.linspace(lat - delta_deg, lat + delta_deg, nb_points)
    lons = np.linspace(lon - delta_deg, lon + delta_deg, nb_points)
    grille_lat, grille_lon = np.meshgrid(lats, lons)
    points = list(zip(grille_lat.ravel(), grille_lon.ravel()))

    url = 'https://api.opentopodata.org/v1/srtm90m'
    altitudes = []

    # L'API accepte max 100 points par requête
    batch_size = 100
    print(f'⏳ Récupération des altitudes pour {len(points)} points...')
    for i in range(0, len(points), batch_size):
        batch = points[i:i + batch_size]
        locations = '|'.join([f'{p[0]:.5f},{p[1]:.5f}' for p in batch])
        try:
            resp = requests.get(url, params={'locations': locations}, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            altitudes.extend([r.get('elevation', 0) or 0 for r in data.get('results', [])])
        except Exception as e:
            print(f'⚠️  Erreur API altitude (batch {i}): {e}')
            altitudes.extend([0] * len(batch))

    # Récupération de l'altitude du point central
    try:
        resp_centre = requests.get(url, params={'locations': f'{lat:.5f},{lon:.5f}'}, timeout=30)
        resp_centre.raise_for_status()
        altitude_centre = resp_centre.json()['results'][0].get('elevation', 0) or 0
    except Exception:
        altitude_centre = altitudes[len(altitudes) // 2] if altitudes else 0

    alt_array = np.array(altitudes).reshape(nb_points, nb_points)
    print(f'✅ Altitude du site : {altitude_centre:.0f} m')
    return alt_array, lats, lons, float(altitude_centre)


alt_grille, lats_grille, lons_grille, altitude_site = recuperer_altitude(
    LATITUDE, LONGITUDE, rayon_km=5, nb_points=25
)

## 10. 🌊 Évaluation du Risque d'Inondation

In [ ]:
def evaluer_risque_inondation(df_meteo, alt_grille, lats, lons, altitude_site):
    """
    Évalue le risque d'inondation en combinant :
    - Précipitations intenses (> 30 mm/j)
    - Altitude du site
    - Pente du terrain (calculée depuis le MNT)
    """
    precip = df_meteo['precipitation'].fillna(0)

    # ---- Indicateur 1 : précipitations extrêmes ----
    jours_extremes = (precip > 30).sum()
    max_precip = precip.max()
    score_precip = min(jours_extremes / 5.0, 1.0)  # normalisé 0–1

    # ---- Indicateur 2 : altitude (zones basses = risque plus élevé) ----
    if altitude_site < 50:
        score_altitude = 1.0
    elif altitude_site < 200:
        score_altitude = 0.6
    elif altitude_site < 500:
        score_altitude = 0.3
    else:
        score_altitude = 0.1

    # ---- Indicateur 3 : pente (zones plates = accumulation d'eau) ----
    dy, dx = np.gradient(alt_grille)
    pente_moy = np.sqrt(dx**2 + dy**2).mean()
    score_pente = max(0, 1 - pente_moy / 10)  # normalisé 0–1 (pente faible = risque élevé)

    # Score global (pondéré)
    score_global = 0.5 * score_precip + 0.3 * score_altitude + 0.2 * score_pente

    # Classification
    if score_global >= 0.7:
        niveau = 'ÉLEVÉ'
        couleur = 'red'
        emoji = '🔴'
    elif score_global >= 0.4:
        niveau = 'MODÉRÉ'
        couleur = 'orange'
        emoji = '🟠'
    elif score_global >= 0.2:
        niveau = 'FAIBLE'
        couleur = 'yellow'
        emoji = '🟡'
    else:
        niveau = 'TRÈS FAIBLE'
        couleur = 'green'
        emoji = '🟢'

    # Visualisation
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('🌊 Analyse du Risque d\'Inondation', fontsize=14, fontweight='bold')

    # MNT avec courbes de niveau
    im = axes[0].imshow(alt_grille, cmap='terrain', origin='lower')
    axes[0].contour(alt_grille, levels=10, colors='black', linewidths=0.5, alpha=0.5)
    axes[0].set_title('Modèle Numérique de Terrain (m)')
    axes[0].set_xlabel('Longitude (index)')
    axes[0].set_ylabel('Latitude (index)')
    plt.colorbar(im, ax=axes[0], label='Altitude (m)')

    # Carte de pente
    pente = np.sqrt(dx**2 + dy**2)
    im2 = axes[1].imshow(pente, cmap='YlOrRd', origin='lower')
    axes[1].set_title('Carte des Pentes')
    axes[1].set_xlabel('Longitude (index)')
    axes[1].set_ylabel('Latitude (index)')
    plt.colorbar(im2, ax=axes[1], label='Pente relative')

    # Résumé risque inondation
    axes[2].axis('off')
    info = [
        (f'{emoji} Niveau de risque : {niveau}', 14, couleur),
        (f'Score global : {score_global:.2f} / 1.00', 11, 'black'),
        ('', 10, 'black'),
        (f'Altitude du site : {altitude_site:.0f} m', 10, 'black'),
        (f'Score altitude : {score_altitude:.2f}', 10, 'gray'),
        (f'Jours pluie > 30 mm : {jours_extremes}', 10, 'black'),
        (f'Score précipitations : {score_precip:.2f}', 10, 'gray'),
        (f'Pente moyenne : {pente_moy:.2f}', 10, 'black'),
        (f'Score pente : {score_pente:.2f}', 10, 'gray'),
        ('', 10, 'black'),
        ('Pondération :', 10, 'black'),
        ('  Précipitations (50%)', 9, 'gray'),
        ('  Altitude (30%)', 9, 'gray'),
        ('  Pente (20%)', 9, 'gray'),
    ]
    y = 0.95
    for texte, taille, col in info:
        axes[2].text(0.05, y, texte, fontsize=taille, color=col,
                     transform=axes[2].transAxes)
        y -= 0.07 if taille >= 11 else 0.06

    plt.tight_layout()
    plt.savefig('/tmp/risque_inondation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n{emoji} Risque d\'inondation : {niveau} (score={score_global:.2f})')
    return {
        'niveau': niveau, 'score': score_global,
        'couleur': couleur, 'emoji': emoji
    }


risque_inondation = evaluer_risque_inondation(
    df_meteo, alt_grille, lats_grille, lons_grille, altitude_site
)

## 11. 🏔️ Évaluation du Risque de Glissement de Terrain

In [ ]:
def evaluer_risque_glissement(df_meteo, alt_grille, altitude_site):
    """
    Évalue le risque de glissement de terrain en combinant :
    - Précipitations soutenues (pluies cumulées sur plusieurs jours)
    - Pente du terrain (forte pente = risque élevé)
    - Altitude (zones de relief = risque plus élevé)
    """
    precip = df_meteo['precipitation'].fillna(0)

    # ---- Indicateur 1 : pluies cumulées sur 5 jours ----
    cumul_5j = precip.rolling(window=5).sum()
    max_cumul_5j = cumul_5j.max()
    nb_episodes = (cumul_5j > 80).sum()  # Épisodes > 80 mm / 5 jours
    score_pluie = min(nb_episodes / 3.0, 1.0)

    # ---- Indicateur 2 : pente du terrain ----
    dy, dx = np.gradient(alt_grille)
    pente = np.sqrt(dx**2 + dy**2)
    pente_moy = pente.mean()
    pente_max = pente.max()
    score_pente = min(pente_moy / 5.0, 1.0)  # pente forte = score élevé

    # ---- Indicateur 3 : relief (altitude > 200 m = risque plus élevé) ----
    if altitude_site < 100:
        score_relief = 0.1
    elif altitude_site < 500:
        score_relief = 0.4
    elif altitude_site < 1000:
        score_relief = 0.7
    else:
        score_relief = 1.0

    # Score global
    score_global = 0.4 * score_pente + 0.35 * score_pluie + 0.25 * score_relief

    if score_global >= 0.7:
        niveau = 'ÉLEVÉ'
        couleur = 'red'
        emoji = '🔴'
    elif score_global >= 0.4:
        niveau = 'MODÉRÉ'
        couleur = 'orange'
        emoji = '🟠'
    elif score_global >= 0.2:
        niveau = 'FAIBLE'
        couleur = 'yellow'
        emoji = '🟡'
    else:
        niveau = 'TRÈS FAIBLE'
        couleur = 'green'
        emoji = '🟢'

    # Visualisation
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('🏔️ Analyse du Risque de Glissement de Terrain', fontsize=14, fontweight='bold')

    # Précipitations cumulées sur 5 jours
    axes[0].plot(cumul_5j.index, cumul_5j.values, color='royalblue', linewidth=1)
    axes[0].axhline(80, color='red', linestyle='--', label='Seuil critique (80 mm/5j)')
    axes[0].fill_between(cumul_5j.index, cumul_5j.values, 80,
                          where=cumul_5j.values > 80, alpha=0.3, color='red', label='Épisodes critiques')
    axes[0].set_title('Précipitations cumulées sur 5 jours')
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Cumul 5 jours (mm)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend()

    # Carte de susceptibilité (pente × précipitations normalisées)
    precip_norm = float(max_cumul_5j) / 200 if max_cumul_5j > 0 else 0
    susceptibilite = pente * min(precip_norm, 1.0)
    im = axes[1].imshow(susceptibilite, cmap='YlOrRd', origin='lower')
    axes[1].set_title('Carte de Susceptibilité\n(pente × précipitations)')
    axes[1].set_xlabel('Longitude (index)')
    axes[1].set_ylabel('Latitude (index)')
    plt.colorbar(im, ax=axes[1], label='Indice de susceptibilité')

    # Résumé
    axes[2].axis('off')
    info = [
        (f'{emoji} Niveau de risque : {niveau}', 14, couleur),
        (f'Score global : {score_global:.2f} / 1.00', 11, 'black'),
        ('', 10, 'black'),
        (f'Altitude du site : {altitude_site:.0f} m', 10, 'black'),
        (f'Score relief : {score_relief:.2f}', 10, 'gray'),
        (f'Épisodes pluie > 80 mm/5j : {nb_episodes}', 10, 'black'),
        (f'Max cumul 5 jours : {max_cumul_5j:.1f} mm', 10, 'black'),
        (f'Score pluie : {score_pluie:.2f}', 10, 'gray'),
        (f'Pente moyenne : {pente_moy:.2f}', 10, 'black'),
        (f'Pente maximale : {pente_max:.2f}', 10, 'black'),
        (f'Score pente : {score_pente:.2f}', 10, 'gray'),
    ]
    y = 0.95
    for texte, taille, col in info:
        axes[2].text(0.05, y, texte, fontsize=taille, color=col,
                     transform=axes[2].transAxes)
        y -= 0.07 if taille >= 11 else 0.06

    plt.tight_layout()
    plt.savefig('/tmp/risque_glissement.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n{emoji} Risque de glissement : {niveau} (score={score_global:.2f})')
    return {
        'niveau': niveau, 'score': score_global,
        'couleur': couleur, 'emoji': emoji
    }


risque_glissement = evaluer_risque_glissement(df_meteo, alt_grille, altitude_site)

## 12. 🌍 Évaluation du Risque Sismique (USGS)

In [ ]:
def evaluer_risque_sismique(lat, lon, rayon_km=200, min_magnitude=3.0, nb_annees=10):
    """
    Récupère et analyse les séismes historiques autour du site
    via l'API USGS Earthquake (gratuite).
    """
    date_fin = datetime.utcnow()
    date_debut = date_fin - timedelta(days=365 * nb_annees)

    url = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
    params = {
        'format': 'geojson',
        'starttime': date_debut.strftime('%Y-%m-%d'),
        'endtime': date_fin.strftime('%Y-%m-%d'),
        'latitude': lat,
        'longitude': lon,
        'maxradiuskm': rayon_km,
        'minmagnitude': min_magnitude,
        'orderby': 'magnitude',
        'limit': 500
    }

    print(f'⏳ Récupération des séismes (rayon={rayon_km} km, M≥{min_magnitude}, {nb_annees} ans)...')
    try:
        resp = requests.get(url, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f'⚠️  Erreur API USGS: {e}')
        return {'niveau': 'INCONNU', 'score': 0, 'couleur': 'gray', 'emoji': '⚫', 'nb_seismes': 0}

    features = data.get('features', [])
    nb_seismes = len(features)
    print(f'✅ {nb_seismes} séismes récupérés')

    if nb_seismes == 0:
        df_seismes = pd.DataFrame(columns=['magnitude', 'depth', 'date', 'lat', 'lon', 'lieu'])
    else:
        records = []
        for f in features:
            props = f['properties']
            coords = f['geometry']['coordinates']
            records.append({
                'magnitude': props.get('mag', 0),
                'depth': coords[2],
                'date': datetime.utcfromtimestamp(props['time'] / 1000),
                'lat': coords[1],
                'lon': coords[0],
                'lieu': props.get('place', 'Inconnu')
            })
        df_seismes = pd.DataFrame(records)

    # Calcul du score sismique
    if nb_seismes == 0:
        score_global = 0.05
    else:
        mag_max = df_seismes['magnitude'].max()
        nb_fort = (df_seismes['magnitude'] >= 5.0).sum()
        nb_moyen = (df_seismes['magnitude'] >= 4.0).sum()
        freq_annuelle = nb_seismes / nb_annees

        score_mag = min((mag_max - 3) / 5, 1.0) if mag_max > 3 else 0
        score_freq = min(freq_annuelle / 20, 1.0)
        score_fort = min(nb_fort / 5, 1.0)
        score_global = 0.4 * score_mag + 0.35 * score_freq + 0.25 * score_fort

    if score_global >= 0.7:
        niveau = 'ÉLEVÉ'
        couleur = 'red'
        emoji = '🔴'
    elif score_global >= 0.4:
        niveau = 'MODÉRÉ'
        couleur = 'orange'
        emoji = '🟠'
    elif score_global >= 0.15:
        niveau = 'FAIBLE'
        couleur = 'yellow'
        emoji = '🟡'
    else:
        niveau = 'TRÈS FAIBLE'
        couleur = 'green'
        emoji = '🟢'

    # ---- Visualisations ----
    if nb_seismes > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle('🌍 Analyse du Risque Sismique', fontsize=14, fontweight='bold')

        # Carte des séismes
        sc = axes[0].scatter(
            df_seismes['lon'], df_seismes['lat'],
            c=df_seismes['magnitude'], cmap='hot_r',
            s=df_seismes['magnitude'] ** 3, alpha=0.6,
            vmin=min_magnitude, vmax=8
        )
        axes[0].plot(lon, lat, 'b^', markersize=12, label='Site d\'étude', zorder=5)
        plt.colorbar(sc, ax=axes[0], label='Magnitude')
        axes[0].set_title(f'Carte sismique (rayon {rayon_km} km)')
        axes[0].set_xlabel('Longitude')
        axes[0].set_ylabel('Latitude')
        axes[0].legend()

        # Distribution des magnitudes
        mags = df_seismes['magnitude']
        axes[1].hist(mags, bins=20, color='tomato', edgecolor='black', alpha=0.75)
        axes[1].axvline(mags.mean(), color='darkred', linestyle='--',
                        label=f'Moy: {mags.mean():.1f}')
        axes[1].axvline(mags.max(), color='black', linestyle=':',
                        label=f'Max: {mags.max():.1f}')
        axes[1].set_title('Distribution des magnitudes')
        axes[1].set_xlabel('Magnitude')
        axes[1].set_ylabel('Nombre de séismes')
        axes[1].legend()

        # Résumé
        axes[2].axis('off')
        info = [
            (f'{emoji} Niveau de risque : {niveau}', 14, couleur),
            (f'Score global : {score_global:.2f} / 1.00', 11, 'black'),
            ('', 10, 'black'),
            (f'Nombre de séismes ({nb_annees} ans) : {nb_seismes}', 10, 'black'),
            (f'Magnitude maximale : {mags.max():.1f}', 10, 'black'),
            (f'Magnitude moyenne : {mags.mean():.1f}', 10, 'black'),
            (f'Séismes M≥5.0 : {(mags >= 5.0).sum()}', 10, 'black'),
            (f'Séismes M≥4.0 : {(mags >= 4.0).sum()}', 10, 'black'),
            (f'Fréquence/an : {nb_seismes / nb_annees:.1f}', 10, 'black'),
        ]
        y = 0.95
        for texte, taille, col in info:
            axes[2].text(0.05, y, texte, fontsize=taille, color=col,
                         transform=axes[2].transAxes)
            y -= 0.08 if taille >= 11 else 0.065

        plt.tight_layout()
        plt.savefig('/tmp/risque_sismique.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('ℹ️  Aucun séisme détecté dans la zone.')

    print(f'\n{emoji} Risque sismique : {niveau} (score={score_global:.2f})')
    return {
        'niveau': niveau, 'score': score_global,
        'couleur': couleur, 'emoji': emoji,
        'nb_seismes': nb_seismes,
        'df': df_seismes if nb_seismes > 0 else None
    }


risque_sismique = evaluer_risque_sismique(LATITUDE, LONGITUDE, rayon_km=200, min_magnitude=3.0)

## 13. 🗺️ Carte des Risques Naturels (Folium)

In [ ]:
def creer_carte_risques(lat, lon, nom_site, risque_inond, risque_glis, risque_sism,
                         alt_grille, lats_grille, lons_grille, df_seismes=None):
    """Crée une carte Folium multicouches avec tous les risques naturels."""

    carte = folium.Map(location=[lat, lon], zoom_start=9, tiles='OpenStreetMap')

    # Couche satellite
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite', overlay=False, control=True
    ).add_to(carte)

    # Groupe : Localisation du site
    grp_site = folium.FeatureGroup(name='📍 Site d\'étude', show=True)
    couleurs = {'ÉLEVÉ': 'red', 'MODÉRÉ': 'orange', 'FAIBLE': 'beige', 'TRÈS FAIBLE': 'green', 'INCONNU': 'gray'}
    popup_html = f"""
    <b>{nom_site}</b><br>
    Latitude: {lat:.4f}°<br>
    Longitude: {lon:.4f}°<br>
    Altitude: {altitude_site:.0f} m<br><br>
    <b>Risques :</b><br>
    🌊 Inondation: {risque_inond['niveau']}<br>
    🏔️ Glissement: {risque_glis['niveau']}<br>
    🌍 Sismique: {risque_sism['niveau']}
    """
    folium.Marker(
        [lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=nom_site,
        icon=folium.Icon(color='red', icon='home', prefix='fa')
    ).add_to(grp_site)
    folium.Circle([lat, lon], radius=500, color='red', fill=True,
                  fill_opacity=0.1).add_to(grp_site)
    grp_site.add_to(carte)

    # Groupe : Données d'altitude (heatmap)
    grp_alt = folium.FeatureGroup(name='🏔️ Altitude (Heatmap)', show=False)
    heat_data = []
    alt_min = alt_grille.min()
    alt_max = alt_grille.max()
    for i, lat_g in enumerate(lats_grille):
        for j, lon_g in enumerate(lons_grille):
            alt_norm = float(alt_grille[j, i] - alt_min) / max(alt_max - alt_min, 1)
            heat_data.append([float(lat_g), float(lon_g), alt_norm])
    HeatMap(heat_data, name='Altitude', min_opacity=0.3, radius=15).add_to(grp_alt)
    grp_alt.add_to(carte)

    # Groupe : Séismes historiques
    if df_seismes is not None and len(df_seismes) > 0:
        grp_seism = folium.FeatureGroup(name='🌍 Séismes historiques', show=True)
        mc = MarkerCluster().add_to(grp_seism)
        for _, row in df_seismes.iterrows():
            mag = row['magnitude']
            color_seis = 'red' if mag >= 5 else 'orange' if mag >= 4 else 'yellow'
            folium.CircleMarker(
                location=[row['lat'], row['lon']],
                radius=max(mag * 2, 4),
                color=color_seis,
                fill=True,
                fill_opacity=0.6,
                popup=folium.Popup(
                    f'<b>M{mag:.1f}</b><br>{row["lieu"]}<br>'
                    f'Profondeur: {row["depth"]:.1f} km<br>{row["date"].strftime("%Y-%m-%d")}',
                    max_width=200
                )
            ).add_to(mc)
        grp_seism.add_to(carte)

    # Cercle de zone d'étude
    for rayon, nom in [(5000, '5 km'), (20000, '20 km'), (50000, '50 km')]:
        folium.Circle(
            [lat, lon], radius=rayon, color='blue', fill=False,
            weight=1, opacity=0.4, tooltip=f'Zone {nom}'
        ).add_to(carte)

    folium.LayerControl(collapsed=False).add_to(carte)
    carte.save('/tmp/carte_risques.html')
    display(carte)
    print('✅ Carte des risques sauvegardée dans /tmp/carte_risques.html')


creer_carte_risques(
    LATITUDE, LONGITUDE, NOM_SITE,
    risque_inondation, risque_glissement, risque_sismique,
    alt_grille, lats_grille, lons_grille,
    df_seismes=risque_sismique.get('df')
)

## 14. 📊 Tableau de Bord de Synthèse

In [ ]:
def afficher_tableau_de_bord(nom_site, lat, lon, altitude, df_meteo,
                              risque_inond, risque_glis, risque_sism, stats_precip):
    """Génère un tableau de bord de synthèse."""

    fig = plt.figure(figsize=(20, 14))
    fig.patch.set_facecolor('#f0f4f8')
    fig.suptitle(f'📊 Rapport Géomatique – {nom_site}\n'
                 f'Lat: {lat:.4f}° | Lon: {lon:.4f}° | Alt: {altitude:.0f} m',
                 fontsize=16, fontweight='bold', y=0.98)

    # ---- Panneau gauche : récapitulatif des risques ----
    ax_risques = fig.add_subplot(3, 4, (1, 5))  # 2 lignes × 1 colonne à gauche
    ax_risques.axis('off')
    ax_risques.set_facecolor('#ffffff')

    risques = [
        ('🌊 Inondation', risque_inond),
        ('🏔️ Glissement', risque_glis),
        ('🌍 Sismique', risque_sism),
    ]
    ax_risques.text(0.5, 0.97, 'NIVEAUX DE RISQUE NATUREL',
                    ha='center', va='top', fontsize=13, fontweight='bold',
                    transform=ax_risques.transAxes)

    color_map = {'ÉLEVÉ': '#e74c3c', 'MODÉRÉ': '#e67e22',
                 'FAIBLE': '#f1c40f', 'TRÈS FAIBLE': '#2ecc71', 'INCONNU': '#95a5a6'}
    y = 0.82
    for nom_risque, risque in risques:
        niveau = risque['niveau']
        score = risque['score']
        bg_color = color_map.get(niveau, '#95a5a6')
        ax_risques.text(0.05, y, f'{nom_risque}:', fontsize=12,
                        transform=ax_risques.transAxes, fontweight='bold')
        ax_risques.text(0.55, y, f'{niveau}', fontsize=12,
                        transform=ax_risques.transAxes,
                        color='white', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.4', facecolor=bg_color, alpha=0.9))
        ax_risques.text(0.05, y - 0.07, f'  Score : {score:.2f}/1.00',
                        fontsize=10, color='gray', transform=ax_risques.transAxes)
        # Barre de score
        bar_x = 0.05
        bar_y = y - 0.12
        ax_risques.barh([bar_y], [score], height=0.04, left=bar_x,
                        color=bg_color, alpha=0.8, transform=ax_risques.transAxes)
        ax_risques.barh([bar_y], [1 - score], height=0.04, left=bar_x + score,
                        color='#ecf0f1', alpha=0.8, transform=ax_risques.transAxes)
        y -= 0.28

    # ---- Températures mensuelles ----
    ax_temp = fig.add_subplot(3, 4, (2, 3))
    temp_m = df_meteo[['temp_min', 'temp_mean', 'temp_max']].resample('ME').mean()
    mois = [d.strftime('%b') for d in temp_m.index]
    x = np.arange(len(mois))
    ax_temp.fill_between(x, temp_m['temp_min'], temp_m['temp_max'], alpha=0.2, color='orange')
    ax_temp.plot(x, temp_m['temp_mean'], 'r-o', ms=4, linewidth=2)
    ax_temp.set_xticks(x)
    ax_temp.set_xticklabels(mois, fontsize=8)
    ax_temp.set_title('🌡️ Températures mensuelles (°C)', fontsize=10)
    ax_temp.set_ylabel('°C')
    ax_temp.grid(alpha=0.3)

    # ---- Précipitations mensuelles ----
    ax_precip = fig.add_subplot(3, 4, 4)
    precip_m = df_meteo['precipitation'].fillna(0).resample('ME').sum()
    mois_p = [d.strftime('%b') for d in precip_m.index]
    ax_precip.bar(range(len(mois_p)), precip_m.values, color='steelblue', alpha=0.8)
    ax_precip.set_xticks(range(len(mois_p)))
    ax_precip.set_xticklabels(mois_p, fontsize=8, rotation=45)
    ax_precip.set_title('🌧️ Précipitations mensuelles (mm)', fontsize=10)
    ax_precip.set_ylabel('mm')
    ax_precip.grid(alpha=0.3, axis='y')

    # ---- Rose des vents compacte ----
    ax_vent = fig.add_subplot(3, 4, (6, 7), projection='polar')
    mask = ~(df_meteo['vent_direction'].isna() | df_meteo['vent_vitesse'].isna())
    dirs = df_meteo.loc[mask, 'vent_direction'].values
    vits = df_meteo.loc[mask, 'vent_vitesse'].values
    if len(dirs) > 0:
        nb_s = 16
        angles_s = np.linspace(0, 360, nb_s, endpoint=False)
        freq_s = np.zeros(nb_s)
        larg = 360 / nb_s
        for j, ang in enumerate(angles_s):
            freq_s[j] = np.sum(
                (dirs >= ang - larg / 2) & (dirs < ang + larg / 2)
            ) / len(dirs) * 100
        ax_vent.set_theta_zero_location('N')
        ax_vent.set_theta_direction(-1)
        ax_vent.bar(np.radians(angles_s), freq_s,
                    width=np.radians(larg), color='steelblue', alpha=0.75)
        ax_vent.set_xticks(np.radians([0, 45, 90, 135, 180, 225, 270, 315]))
        ax_vent.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SO', 'O', 'NO'], fontsize=8)
    ax_vent.set_title('💨 Rose des vents', fontsize=10, pad=15)

    # ---- Statistiques clés ----
    ax_stats = fig.add_subplot(3, 4, 8)
    ax_stats.axis('off')
    precip = df_meteo['precipitation'].fillna(0)
    temp = df_meteo['temp_mean'].dropna()
    vent_v = df_meteo['vent_vitesse'].dropna()
    lines = [
        '📊 STATISTIQUES CLÉS',
        '',
        f'🌡️ T° moyenne : {temp.mean():.1f}°C',
        f'🌡️ T° max : {df_meteo["temp_max"].max():.1f}°C',
        f'🌡️ T° min : {df_meteo["temp_min"].min():.1f}°C',
        '',
        f'🌧️ Pluie totale : {precip.sum():.0f} mm/an',
        f'🌧️ Max/jour : {precip.max():.0f} mm',
        f'🌧️ Jours pluie : {(precip > 0).sum()}',
        '',
        f'💨 Vent moy : {vent_v.mean():.1f} km/h',
        f'💨 Vent max : {vent_v.max():.1f} km/h',
        '',
        f'🌍 Nb séismes : {risque_sism["nb_seismes"]}',
    ]
    y_s = 0.97
    for line in lines:
        weight = 'bold' if line.startswith('📊') else 'normal'
        ax_stats.text(0.05, y_s, line, fontsize=9, transform=ax_stats.transAxes,
                      fontweight=weight)
        y_s -= 0.065

    # ---- Légende des niveaux de risque ----
    ax_legende = fig.add_subplot(3, 4, (9, 12))
    ax_legende.axis('off')
    legende_items = [
        ('🔴 ÉLEVÉ', '#e74c3c', 'Risque important – mesures urgentes requises'),
        ('🟠 MODÉRÉ', '#e67e22', 'Risque notable – surveillance recommandée'),
        ('🟡 FAIBLE', '#f1c40f', 'Risque limité – vigilance normale'),
        ('🟢 TRÈS FAIBLE', '#2ecc71', 'Risque négligeable – conditions favorables'),
    ]
    ax_legende.text(0.5, 0.95, 'LÉGENDE DES NIVEAUX DE RISQUE',
                    ha='center', fontsize=11, fontweight='bold',
                    transform=ax_legende.transAxes)
    y_l = 0.75
    for label, col, desc in legende_items:
        ax_legende.text(0.02, y_l, label, fontsize=11, fontweight='bold', color=col,
                        transform=ax_legende.transAxes)
        ax_legende.text(0.22, y_l, f': {desc}', fontsize=10,
                        transform=ax_legende.transAxes)
        y_l -= 0.18

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('/tmp/tableau_de_bord.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Tableau de bord généré et sauvegardé')


afficher_tableau_de_bord(
    NOM_SITE, LATITUDE, LONGITUDE, altitude_site,
    df_meteo, risque_inondation, risque_glissement, risque_sismique, stats_precip
)

## 15. 📄 Rapport Textuel de Synthèse

In [ ]:
def generer_rapport(nom_site, lat, lon, altitude, df_meteo,
                    risque_inond, risque_glis, risque_sism, stats_precip):
    """Génère un rapport textuel de synthèse."""

    precip = df_meteo['precipitation'].fillna(0)
    temp = df_meteo['temp_mean'].dropna()
    vent_v = df_meteo['vent_vitesse'].dropna()
    vent_d = df_meteo['vent_direction'].dropna()

    # Direction dominante du vent
    dir_dominante_deg = vent_d.mode()[0] if len(vent_d) > 0 else 0
    points_cardinaux = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                        'S', 'SSO', 'SO', 'OSO', 'O', 'ONO', 'NO', 'NNO']
    idx_cardinal = round(dir_dominante_deg / 22.5) % 16
    dir_dominante_nom = points_cardinaux[idx_cardinal]

    rapport = f"""
{'='*70}
  RAPPORT D'ANALYSE GÉOMATIQUE
{'='*70}

SITE D'ÉTUDE    : {nom_site}
Coordonnées     : {lat:.4f}°N, {lon:.4f}°E
Altitude        : {altitude:.0f} m
Période analysée: {DATE_DEBUT} → {DATE_FIN}
Date du rapport : {datetime.utcnow().strftime('%Y-%m-%d %H:%M')} UTC

{'─'*70}
1. ANALYSE DES PRÉCIPITATIONS
{'─'*70}
  • Total annuel          : {precip.sum():.1f} mm
  • Moyenne journalière   : {precip.mean():.2f} mm/j
  • Maximum journalier    : {precip.max():.1f} mm
  • Nombre de jours pluie : {(precip > 0).sum()}
  • Jours > 10 mm         : {(precip > 10).sum()}
  • Jours > 30 mm (extr.) : {(precip > 30).sum()}

{'─'*70}
2. ANALYSE DES TEMPÉRATURES
{'─'*70}
  • Température annuelle moyenne : {temp.mean():.1f} °C
  • Température maximale         : {df_meteo['temp_max'].max():.1f} °C
  • Température minimale         : {df_meteo['temp_min'].min():.1f} °C
  • Amplitude thermique annuelle : {df_meteo['temp_max'].max() - df_meteo['temp_min'].min():.1f} °C
  • Jours de gel (T < 0°C)       : {(df_meteo['temp_min'] < 0).sum()}
  • Jours caniculaires (T > 35°C): {(df_meteo['temp_max'] > 35).sum()}

{'─'*70}
3. ANALYSE DES VENTS
{'─'*70}
  • Vitesse moyenne              : {vent_v.mean():.1f} km/h
  • Vitesse maximale             : {vent_v.max():.1f} km/h
  • Percentile 90                : {vent_v.quantile(0.9):.1f} km/h
  • Direction dominante          : {dir_dominante_nom} ({dir_dominante_deg:.0f}°)

{'─'*70}
4. RISQUES NATURELS
{'─'*70}

  🌊 RISQUE D'INONDATION  : {risque_inond['emoji']} {risque_inond['niveau']} (score={risque_inond['score']:.2f})
    Facteurs pris en compte :
    - Précipitations extrêmes (> 30 mm/j) [poids 50%]
    - Altitude du site                    [poids 30%]
    - Pente du terrain                    [poids 20%]

  🏔️ RISQUE DE GLISSEMENT : {risque_glis['emoji']} {risque_glis['niveau']} (score={risque_glis['score']:.2f})
    Facteurs pris en compte :
    - Pente du terrain                    [poids 40%]
    - Épisodes de pluies soutenues (5j)   [poids 35%]
    - Relief (altitude)                   [poids 25%]

  🌍 RISQUE SISMIQUE      : {risque_sism['emoji']} {risque_sism['niveau']} (score={risque_sism['score']:.2f})
    Facteurs pris en compte :
    - Magnitude maximale historique       [poids 40%]
    - Fréquence des séismes               [poids 35%]
    - Séismes M ≥ 5.0                     [poids 25%]
    - Nombre de séismes recensés          : {risque_sism['nb_seismes']}

{'─'*70}
5. RECOMMANDATIONS
{'─'*70}"""

    recommandations = []
    if risque_inond['niveau'] in ('ÉLEVÉ', 'MODÉRÉ'):
        recommandations.append(
            '  ⚠️  Inondation : Prévoir des systèmes de drainage, éviter les constructions '
            'en zones basses, installer des systèmes d\'alerte précoce.'
        )
    if risque_glis['niveau'] in ('ÉLEVÉ', 'MODÉRÉ'):
        recommandations.append(
            '  ⚠️  Glissement : Réaliser une étude géotechnique, renforcer les talus, '
            'planter des végétaux pour stabiliser les pentes.'
        )
    if risque_sism['niveau'] in ('ÉLEVÉ', 'MODÉRÉ'):
        recommandations.append(
            '  ⚠️  Sismique : Appliquer les normes parasismiques de construction, '
            'réaliser une étude de microzonation sismique.'
        )
    if not recommandations:
        recommandations.append('  ✅ Risques globalement faibles – maintenir une surveillance standard.')

    rapport += '\n' + '\n'.join(recommandations)
    rapport += f"\n\n{'='*70}\n  Sources : Open-Meteo, OpenTopoData, USGS Earthquake API\n{'='*70}\n"

    print(rapport)

    # Sauvegarde dans un fichier texte
    with open('/tmp/rapport_geomatique.txt', 'w', encoding='utf-8') as f:
        f.write(rapport)
    print('\n📁 Rapport sauvegardé dans /tmp/rapport_geomatique.txt')
    return rapport


rapport_final = generer_rapport(
    NOM_SITE, LATITUDE, LONGITUDE, altitude_site,
    df_meteo, risque_inondation, risque_glissement, risque_sismique, stats_precip
)

## 16. 💾 Export des fichiers générés

In [ ]:
# Export CSV des données météo
df_meteo.to_csv('/tmp/donnees_meteo.csv', encoding='utf-8')

# Export CSV des séismes (si disponibles)
if risque_sismique.get('df') is not None:
    risque_sismique['df'].to_csv('/tmp/seismes.csv', index=False, encoding='utf-8')

print('📁 Fichiers générés :')
print('  /tmp/carte_site.html        – Carte interactive du site')
print('  /tmp/carte_risques.html     – Carte interactive des risques')
print('  /tmp/precipitations.png     – Graphique des précipitations')
print('  /tmp/rose_des_vents.png     – Rose des vents')
print('  /tmp/temperatures.png       – Analyse des températures')
print('  /tmp/risque_inondation.png  – Analyse risque inondation')
print('  /tmp/risque_glissement.png  – Analyse risque glissement')
print('  /tmp/risque_sismique.png    – Analyse risque sismique')
print('  /tmp/tableau_de_bord.png    – Tableau de bord de synthèse')
print('  /tmp/rapport_geomatique.txt – Rapport textuel complet')
print('  /tmp/donnees_meteo.csv      – Données météo (CSV)')
if risque_sismique.get('df') is not None:
    print('  /tmp/seismes.csv            – Données sismiques (CSV)')

# Téléchargement automatique des fichiers dans Colab
try:
    from google.colab import files
    import os
    fichiers_a_telecharger = [
        '/tmp/carte_risques.html',
        '/tmp/tableau_de_bord.png',
        '/tmp/rapport_geomatique.txt',
        '/tmp/donnees_meteo.csv',
    ]
    for f in fichiers_a_telecharger:
        if os.path.exists(f):
            files.download(f)
    print('\n✅ Fichiers principaux en cours de téléchargement...')
except ImportError:
    print('\nℹ️  Exécution hors Google Colab – les fichiers sont dans /tmp/')